# 10 ChIP Peak Heatmaps

deeptools heatmaps of CUT&RUN Runx3/Runx1 bigWig signal over peaks defined by the Runx3 ChIP (`source_data/Runx3.bed`; see `10_chipOverlap.ipynb`). Samples/bigWigs match `01_peakEDA.ipynb`.

## 10.1 Initialization

In [1]:
docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -e MPLCONFIGDIR=/home/dalbao/.config/matplotlib \
        -v /tmp:/tmp \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
mkdir -p 10_chipPeakHeatmaps


plotHeatmap 3.5.6


Associative array of bigWig files, same samples and order as `01_peakEDA.ipynb`.

In [2]:
declare -A bigWigFiles

for group in shCd19 shRunx3 memory early late terminal; do
    for target in Runx3 Runx1; do
        fn=${group}_${target}_log2.bigWig
        fpath="source_data/bg_corrected_bigWigs/"
        bigWigFiles[${group}_${target}]="${fpath}${fn}"
    done
done

echo Sample: ${bigWigFiles["shCd19_Runx3"]}


Sample: source_data/bg_corrected_bigWigs/shCd19_Runx3_log2.bigWig


## 10.2 ChIP Peaks

Peaks defined by the Runx3 ChIP-seq (`source_data/Runx3.bed`).

In [3]:
chipPeaks="source_data/Runx3.bed"
wc -l "$chipPeaks"
head -n 3 "$chipPeaks"


5673 source_data/Runx3.bed


1	4748290	4748371	Runx3_5460


1	6467244	6467325	Runx3_5461


1	7147118	7147199	Runx3_5462


## 10.3 Compute Matrix over ChIP Peaks

In [4]:
deeptools computeMatrix reference-point \
    -S \
    ${bigWigFiles["shCd19_Runx3"]} \
    ${bigWigFiles["shRunx3_Runx3"]} \
    ${bigWigFiles["shCd19_Runx1"]} \
    ${bigWigFiles["shRunx3_Runx1"]} \
    ${bigWigFiles["memory_Runx3"]} \
    ${bigWigFiles["early_Runx3"]} \
    ${bigWigFiles["late_Runx3"]} \
    ${bigWigFiles["terminal_Runx3"]} \
    ${bigWigFiles["memory_Runx1"]} \
    ${bigWigFiles["early_Runx1"]} \
    ${bigWigFiles["late_Runx1"]} \
    ${bigWigFiles["terminal_Runx1"]} \
    -R \
    "$chipPeaks" \
    --referencePoint center \
    -b 1500 -a 1500 \
    --numberOfProcessors 36 \
    --sortUsing mean \
    -out 10_chipPeakHeatmaps/chipPeaks.gz

ls -1 10_chipPeakHeatmaps/chipPeaks.gz


The following chromosome names did not match between the bigwig files


chromosome	length


              Y	  91744698


10_chipPeakHeatmaps/chipPeaks.gz


## 10.4 Heatmap — No Clustering

In [5]:
deeptools plotHeatmap \
    -m "10_chipPeakHeatmaps/chipPeaks.gz" \
    -out "10_chipPeakHeatmaps/chipPeaks.pdf" \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --regionsLabel "Runx3 ChIP peaks" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1"

ls -1 10_chipPeakHeatmaps/chipPeaks.pdf


10_chipPeakHeatmaps/chipPeaks.pdf


## 10.5 Heatmap — k-means Clustering (k=2)

In [6]:
deeptools plotHeatmap \
    -m "10_chipPeakHeatmaps/chipPeaks.gz" \
    -out "10_chipPeakHeatmaps/chipPeaks.k2.pdf" \
    --kmeans 2 \
    --sortUsing sum \
    --colorMap "RdYlBu_r" \
    --samplesLabel "shCd19_Runx3" "shRunx3_Runx3" "shCd19_Runx1" "shRunx3_Runx1" \
                    "memory_Runx3" "early_Runx3" "late_Runx3" "terminal_Runx3" \
                    "memory_Runx1" "early_Runx1" "late_Runx1" "terminal_Runx1" \
    --outFileSortedRegions "10_chipPeakHeatmaps/chipPeaks.k2.sorted.bed"

ls -1 10_chipPeakHeatmaps/chipPeaks.k2.pdf


*Warning* For clustering nan values have to be replaced by zeros 


10_chipPeakHeatmaps/chipPeaks.k2.pdf
